# LoRA and full fine-tuning

What an adapter trains and why memory includes more than weights.

> All examples run on CPU without model downloads and use synthetic data. They explain mechanisms; real training quality must be evaluated separately.

## What to understand

LoRA adds a low-rank update B×A to frozen weights W. A commonly starts random and B starts at zero, preserving the initial function while allowing a nonzero B gradient. The alpha/r scaling affects the update.

Freezing reduces gradient and optimizer memory; base weights and activations remain. QLoRA also quantizes the base and needs a compatible backend. This CPU exercise demonstrates LoRA without quantization or Unsloth.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Run the experiment

Run cells in order. Edit the code; Shift + Enter runs the current cell.

In [ ]:
base = nn.Linear(12, 8); base.requires_grad_(False)
rank, alpha = 2, 4
A = nn.Parameter(torch.randn(rank, 12) * 0.02)
B = nn.Parameter(torch.zeros(8, rank))
optimizer = torch.optim.AdamW([A, B], lr=0.03)
x = torch.randn(32, 12); target = torch.randn(32, 8)
frozen = base.weight.detach().clone()
history = []
for step in range(50):
    prediction = base(x) + (x @ A.T @ B.T) * alpha / rank
    loss = F.mse_loss(prediction, target)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
    history.append(loss.item())
assert torch.equal(base.weight, frozen)
print('trainable:', A.numel() + B.numel(), 'base:', sum(p.numel() for p in base.parameters()))
plt.plot(history, color='#d73535'); plt.ylabel('LoRA MSE'); plt.show()

## Try it yourself

Try ranks 1 and 8, rerunning the whole cell. Compare parameter counts and final error. What happens if both A and B start at zero?

Save a notebook copy and record what you changed, expected, and measured.

## Continue in Studio

The workspace link above the lesson opens related tools. This notebook does not launch large training jobs. Before real training, check data, resources, and method support for the chosen model.

## Sources

- [LoRA](https://arxiv.org/abs/2106.09685)
- [QLoRA](https://arxiv.org/abs/2305.14314)